In [16]:
from cogent3.evolve.models import _general_preds
from cogent3.evolve.predicate import MotifChange, UserPredicate

# to build omega for both trinuc and codon
from cogent3 import get_code
from cogent3.evolve.substitution_model import _CodonPredicates
import typing

In [40]:
def make_gn_preds():
    # making the model parameters (predictates) for
    # the General Nucleotide Markov model
    return _general_preds

print(make_gn_preds())

def make_nr_cpg_preds_strand_symetric():
    # same CpG deamination rate on both strands
    # so one parameter
    return [
        # | is the binary or operator that combines the predicates
        # so we have the union of the two changes as a single parameter
        MotifChange("CG", "TG", forward_only=True)
        | MotifChange("CG", "CA", forward_only=True)
    ]

print(make_nr_cpg_preds_strand_symetric())

def make_omega_preds():
    # making the model parameters (predictates) for
    # the omega parameter of codon models
    gc = get_code(1)
    codon_preds = _CodonPredicates(gc)
    return UserPredicate(codon_preds.replacement).aliased("omega")


omega = [make_omega_preds()]
omega

[A>C, A>T, A>G, C>A, C>T, C>G, T>A, T>C, G>A, G>C, G>T]
[(CG>TG | CG>CA)]


[omega]

Why is cpg decay=omega?

In [41]:
ssym_preds = make_gn_preds() + make_nr_cpg_preds_strand_symetric() + omega
#print(ssym_preds)
cpgdecay = ssym_preds[-1]
cpgdecay

omega

In [42]:
~cpgdecay

~(omega)

How do I get a combined CpG, not CpG ENS estimate? Leaving the function with no predicate doesn't work.

In [37]:
import cogent3
from cogent3.maths.matrix_exponential_integration import expected_number_subs
import paths

import pickle

import trinuc_models as trinucs # this module must be in the same directory as this notebook



region = "intergenicAR/chrm22/sm_output/"
folder_in = paths.DATA_HUMCHIMPORANG115 + region
file_in = folder_in + "trinuc_lh.pickle"
with open(file_in, mode = "rb") as infile: 
    result_IGAR=pickle.load(infile)

humanENS_IGAR = result_IGAR.lf.get_scaled_lengths()['Human']
humanENS_IGAR_cpg = result_IGAR.lf.get_scaled_lengths("CpG")['Human']
humanENS_IGAR_notcpg = result_IGAR.lf.get_scaled_lengths("notCpG")['Human']

print("total ENS:", humanENS_IGAR)
print("cpg ENS:", humanENS_IGAR_cpg)
print("noncpg ENS:", humanENS_IGAR_notcpg)


result_IGAR.lf

TypeError: LikelihoodFunction.get_scaled_lengths() missing 1 required positional argument: 'predicate'

Should I delete te scales and fit the data again?

In [32]:
from cogent3.evolve.ns_substitution_model import (
    NonReversibleCodon,
    NonReversibleTrinucleotide,
)

def _make_model(cls, **kwargs):
    return cls(**kwargs)

def GT_CpG_ss(**kwargs):
    """return a Trinucleotide model with GN predicates, omega, and strand
    symmetric CpG deamination

    Notes
    -----
    This is identical to GNC_CpG_ss except that it has
    ALL possible trinucleotides as states, rather than just the
    sense codons.
    """
    ssym_preds = make_gn_preds() + make_nr_cpg_preds_strand_symetric() + omega
    #print(ssym_preds)
    cpgdecay = ssym_preds[-1]
    kwargs = dict(
        predicates=ssym_preds,
        optimise_motif_probs=False,
        name="GT_CpG_ss",
        scales={"CpG": cpgdecay, "notCpG": ~cpgdecay},
    )
    return _make_model(NonReversibleTrinucleotide, **kwargs)

In [33]:
GT_CpG_ss()

NonReversibleTrinucleotide(name='GT_CpG_ss'; params=['A>C', 'A>T', 'A>G', 'C>A', 'C>T', 'C>G', 'T>A', 'T>C', 'G>A', 'G>C', 'G>T', '(CG>TG | CG>CA)', 'omega']; num_motifs=64; motifs=['TTT', 'TTC', 'TTA', 'TTG', 'TCT', 'TCC', 'TCA', 'TCG', 'TAT', 'TAC', 'TAA', 'TAG', 'TGT', 'TGC', 'TGA', 'TGG', 'CTT', 'CTC', 'CTA', 'CTG', 'CCT', 'CCC', 'CCA', 'CCG', 'CAT', 'CAC', 'CAA', 'CAG', 'CGT', 'CGC', 'CGA', 'CGG', 'ATT', 'ATC', 'ATA', 'ATG', 'ACT', 'ACC', 'ACA', 'ACG', 'AAT', 'AAC', 'AAA', 'AAG', 'AGT', 'AGC', 'AGA', 'AGG', 'GTT', 'GTC', 'GTA', 'GTG', 'GCT', 'GCC', 'GCA', 'GCG', 'GAT', 'GAC', 'GAA', 'GAG', 'GGT', 'GGC', 'GGA', 'GGG']))